# 14_weaviate: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/14_weaviate/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Trace how hybrid alpha changes the first product.

Run: uv run python modules/14_weaviate/examples/01_from_scratch.py
Concepts: Trace how hybrid alpha changes the first product.
Expected: alpha changes the winning title.
CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
"""


def combine(dense, lexical, alpha):
    if not 0 <= alpha <= 1:
        raise ValueError("alpha outside [0,1]")

    def scale(values):
        lo, hi = min(values), max(values)
        return [(v - lo) / (hi - lo) if hi > lo else 0 for v in values]

    return [
        alpha * vector + (1 - alpha) * term
        for vector, term in zip(scale(dense), scale(lexical), strict=True)
    ]


def main() -> None:
    # KEY IDEA: Trace how hybrid alpha changes the first product.
    titles = ["SKU exact match", "semantic alternative", "unrelated"]
    for alpha in [0, 0.5, 1]:
        scores = combine([0.35, 0.95, 0.1], [8, 2, 0], alpha)
        print(alpha, titles[scores.index(max(scores))], scores)
    assert combine([0, 1], [1, 0], 1) == [0, 1]


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/14_weaviate/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("weaviate") is None or "weaviate" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra weaviate; start weaviate and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Query a self-provided Weaviate vector and change the hybrid weight.

    Run: uv run --extra weaviate python modules/14_weaviate/examples/02_production.py
    Concepts: Query a self-provided Weaviate vector and change the hybrid weight.
    Expected: native near-vector filter and alpha sweep.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the weaviate profile first.
    """

    # course-extra: weaviate
    # course-profile: weaviate
    from weaviate.classes.query import Filter, MetadataQuery

    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Query a self-provided Weaviate vector and change the hybrid weight.
        with session("weaviate", 160) as (store, rows, q):
            term = rows.iloc[0].sku
            key = next(iter(store.catalog))
            by_id = store.table.query.near_vector(
                q.tolist(),
                filters=Filter.by_id().equal(key),
                limit=1,
                return_metadata=MetadataQuery(distance=True),
            )
            assert str(by_id.objects[0].uuid) == key
            for alpha in (0.0, 0.5, 1.0):
                ranked = store.table.query.hybrid(
                    query=term,
                    vector=q.tolist(),
                    alpha=alpha,
                    query_properties=["body"],
                    limit=5,
                    return_metadata=MetadataQuery(score=True),
                )
                print("alpha", alpha, [(str(o.uuid), o.metadata.score) for o in ranked.objects])
                assert ranked.objects

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "weaviate")

## 03_measure

Source: `modules/14_weaviate/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("weaviate") is None or "weaviate" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra weaviate; start weaviate and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Measure a controlled database comparison on the same catalog.

    Run: uv run --extra weaviate python modules/14_weaviate/examples/03_measure.py
    Concepts: Measure a controlled database comparison on the same catalog.
    Expected: judged hybrid workloads.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the weaviate profile first.
    """

    # course-extra: weaviate
    # course-profile: weaviate
    from common.tier3 import measure

    def main() -> None:
        # KEY IDEA: Measure a controlled database comparison on the same catalog.
        for alpha in (0.0, 0.5, 1.0):
            measure(
                "weaviate",
                retrieval="hybrid",
                parameters={"alpha": alpha},
                query_types=["keyword", "semantic"],
            )

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "weaviate")

## 04_tenants_generation

Source: `modules/14_weaviate/examples/04_tenants_generation.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("weaviate") is None or "weaviate" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra weaviate; start weaviate and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Fence identical object IDs into separate Weaviate tenants and inspect modules.

    Run: uv run --extra weaviate python modules/14_weaviate/examples/04_tenants_generation.py
    Concepts: Fence identical object IDs into separate Weaviate tenants and inspect modules.
    Expected: tenant-specific titles.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the weaviate profile first.
    """

    # course-extra: weaviate
    # course-profile: weaviate
    import uuid

    from weaviate.classes.config import Configure, DataType, Property
    from weaviate.classes.tenants import Tenant

    from common.stores.weaviate import WeaviateStore

    def main() -> None:
        # KEY IDEA: Fence identical object IDs into separate Weaviate tenants and inspect modules.
        adapter = WeaviateStore()
        name = adapter.name
        db = adapter.db
        try:
            tab = db.collections.create(
                name,
                vector_config=Configure.Vectors.self_provided(),
                multi_tenancy_config=Configure.multi_tenancy(enabled=True),
                properties=[Property(name="title", data_type=DataType.TEXT)],
            )
            try:
                tab.tenants.create([Tenant(name="north"), Tenant(name="south")])
                key = uuid.uuid4()
                for region in ("north", "south"):
                    tab.with_tenant(region).data.insert(
                        uuid=key, properties={"title": region + " mug"}, vector=[1.0, 0.0]
                    )
                for region in ("north", "south"):
                    item = tab.with_tenant(region).query.fetch_object_by_id(key)
                    assert item.properties["title"] == region + " mug"
                modules = sorted(db.get_meta().get("modules", {}))
                print("tenants isolate the same UUID; configured model modules", modules)
                print(
                    "Generative queries require a configured provider and its local model endpoint."
                )
            finally:
                db.collections.delete(name)
        finally:
            db.close()

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "weaviate")

## 05_backup

Source: `modules/14_weaviate/examples/05_backup.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("weaviate") is None or "weaviate" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra weaviate; start weaviate and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Create a filesystem backup and restore its original Weaviate collection.

    Run: uv run --extra weaviate python modules/14_weaviate/examples/05_backup.py
    Concepts: Create a filesystem backup and restore its original Weaviate collection.
    Expected: filesystem backup and restored IDs.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the weaviate profile first.
    """

    # course-extra: weaviate
    # course-profile: weaviate
    import uuid

    from common.tier3 import session

    def main() -> None:
        # KEY IDEA: Create a filesystem backup and restore its original Weaviate collection.
        with session("weaviate", 80) as (s, _, q):
            expected = [h.id for h in s.search(q, 3)]
            backup = "marlowe-" + uuid.uuid4().hex
            s.db.backup.create(
                backup_id=backup,
                backend="filesystem",
                include_collections=[s.name],
                wait_for_completion=True,
            )
            s.db.collections.delete(s.name)
            s.created = False
            s.db.backup.restore(
                backup_id=backup,
                backend="filesystem",
                include_collections=[s.name],
                wait_for_completion=True,
            )
            s.created = True
            s.table = s.db.collections.use(s.name)
            assert [h.id for h in s.search(q, 3)] == expected
            print("restored backup", backup, "IDs", expected)

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "weaviate")

## 06_local_generation

Source: `modules/14_weaviate/examples/06_local_generation.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("weaviate") is None or "weaviate-generation" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra weaviate; start weaviate-generation and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Pass a retrieved product to Weaviate's local generative module.

    Run: uv run --extra weaviate python modules/14_weaviate/examples/06_local_generation.py
    Concepts: Pass a retrieved product to Weaviate's local generative module.
    Expected: optional generated title.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; start the weaviate-generation profile first.
    """

    # course-extra: weaviate
    # course-profile: weaviate-generation
    import uuid

    import weaviate
    from weaviate.classes.config import Configure, DataType, Property
    from weaviate.config import AdditionalConfig, Timeout

    from common.settings import setting
    from common.teaching import evidence

    def main() -> None:
        # KEY IDEA: Pass a retrieved product to Weaviate's local generative module.
        host = setting("WEAVIATE_HOST", "127.0.0.1")
        with weaviate.connect_to_local(
            host=host,
            port=int(setting("GENERATION_PORT", setting("WEAVIATE_PORT", "8080"))),
            grpc_port=int(setting("GENERATION_GRPC_PORT", setting("WEAVIATE_GRPC_PORT", "50051"))),
            skip_init_checks=True,
            additional_config=AdditionalConfig(timeout=Timeout(query=150)),
        ) as db:
            name = "MarloweGen" + uuid.uuid4().hex
            tab = db.collections.create(
                name,
                vector_config=Configure.Vectors.self_provided(),
                generative_config=Configure.Generative.ollama(
                    api_endpoint=setting("GENERATION_OLLAMA_URL", "http://ollama:11434"),
                    model="marlowe-caption",
                ),
                properties=[Property(name="title", data_type=DataType.TEXT)],
            )
            try:
                tab.data.insert(properties={"title": "Coral ceramic mug warmer"}, vector=[1.0, 0.0])
                result = tab.generate.near_vector(
                    near_vector=[1.0, 0.0],
                    single_prompt="Return only the product name from this listing: {title}. /no_think",
                    limit=1,
                )
                obj = result.objects[0]
                assert obj.generated and "mug" in obj.generated.lower()
                print("retrieved title", obj.properties["title"], "generated", obj.generated)
                evidence(
                    "weaviate-local-generation",
                    {
                        "model": "marlowe-caption",
                        "base": "qwen3:0.6b",
                        "source": obj.properties["title"],
                        "response": obj.generated,
                    },
                    tier=3,
                )
            finally:
                db.collections.delete(name)

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "weaviate-generation")

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.teaching import catalog

rows, _, _, _ = catalog(100)
print(rows.groupby("region").price.median())
# Change the group or price threshold, then repeat the native comparison for 14_weaviate.

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.